## Import

In [119]:
import pandas as pd
import psycopg as pg
import os
import json as js
from dotenv import load_dotenv

## Connexion à la DB

In [ ]:
load_dotenv()

connection = pg.connect(
    host="db",
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    port=os.getenv("PORT_DB"),
)

## Conversion en Dataframe

In [121]:
df_poste = pd.read_sql("SELECT * FROM poste", con=connection)

print(df_poste.head())

   id_poste  id_entreprise type_contrat                    date_creation  \
0         4            195    permanent 2024-06-04 10:55:09.704000+00:00   
1         5            196    permanent 2026-04-15 09:06:11.917000+00:00   
2         6            197    permanent 2024-08-14 09:53:35.490000+00:00   
3         7            198    permanent 2025-11-14 17:00:40.361000+00:00   
4         8            198    permanent 2025-11-14 17:00:55.060000+00:00   

                                         description langues  remote_policy  \
0  Sensefuel\nFondée en 2017 par Christophe et St...      fr            3.0   
1  Pour renforcer nos équipes, nous recherchons u...      fr            2.0   
2  Concepteur-développeur confirmé, vous disposez...      fr            2.0   
3   **Rejoins Mergify en tant que Principal Softw...    None            NaN   
4  🎯L'entrepriseCYIM fondée en 2001, se positionn...    None            2.0   

  frequency_remote  annee_experience salaire_currency  ...  \
0     

/tmp/ipykernel_18131/1501290668.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_poste = pd.read_sql("SELECT * FROM poste", con=connection)


## Traitement des données

In [ ]:
# je calcule le nombre de candidatures totales
sum_candidature = int(df_poste["nb_postulations"].sum())
print("Nombre de candidatures total :", sum_candidature, "\n")

# je groupe par categorie
grouped_poste = df_poste.groupby("categorie", as_index=False)["nb_postulations"].sum()

# je convertis la colonne nb_postulation en entier
grouped_poste["nb_postulations"] = grouped_poste["nb_postulations"].astype(int)

# j'ajoute la colonne pour calculer la part des candidatures
grouped_poste["part_candidatures"] = (grouped_poste["nb_postulations"] * 100) / sum_candidature

# je trie par ordre décroissant pour avoir les plus importantes
grouped_poste = grouped_poste.sort_values(["part_candidatures"], ascending=False).reset_index(drop=True)

# j'ajoute les catégories qui possèdent plus de 10% des parts
# s'il n'y en a aucune, alors j'ajoute le top 5
# s'il y en a plus que 5, je garde le top 5
dictionnaire = {}

for index, ligne in grouped_poste.iterrows():
    if ligne["part_candidatures"] >= 10:
        dictionnaire[ligne["categorie"]] = round(ligne["part_candidatures"], 2)
    else:
        break

if len(dictionnaire) == 0:
    for index, ligne in grouped_poste.head(5).iterrows():
        dictionnaire[ligne["categorie"]] = round(ligne["part_candidatures"], 2)

if len(dictionnaire) > 5 :
    dictionnaire = dict(list(dictionnaire.items())[:5])

# je calcule la part des autres catégories
pourcentage = 0
for clé, valeur in dictionnaire.items():
    pourcentage += valeur

dictionnaire["Autres"] = round((100 - pourcentage), 2)

# résultat
print(dictionnaire)

Nombre de candidatures total : 2241 

{'fullstack': 61.49, 'backend': 18.34, 'Autres': 20.17}


/tmp/ipykernel_18131/2305013723.py:6: FutureWarning: The provided callable <built-in function sum> is currently using np.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string np.sum instead.
  grouped_poste = df_poste.groupby("categorie", as_index=False)["nb_postulations"].apply(sum)


## Conversion en liste exploitable par le front

In [ ]:
json = []

total = {
    "total": sum_candidature
}
json.append(total)

couleurs_2 = ["#47AD95", "#414141"]
couleurs_3 = ["#47AD95", "#8208D4", "#414141"]
couleurs_4 = ["#47AD95", "#2A9EBD", "#8208D4", "#414141"]
couleurs_5 = ["#47AD95", "#2A9EBD", "#214CC4", "#8208D4", "#414141"]
couleurs_6 = ["#47AD95", "#2A9EBD", "#214CC4", "#8208D4", "#5B0C83", "#414141"]

palette_map = {
    2: couleurs_2,
    3: couleurs_3,
    4: couleurs_4,
    5: couleurs_5,
    6: couleurs_6
}

for i, (clé, valeur) in enumerate(dictionnaire.items()):
    element = {
        "label": clé,
        "value": valeur,
        "color": palette_map[len(dictionnaire)][i]
    }
    json.append(element)

if not os.path.exists("../../Frontend/techyourjob-frontend/public/data/cache"):
    os.makedirs("../../Frontend/techyourjob-frontend/public/data/cache")

frontend_cache_path = "../../Frontend/techyourjob-frontend/public/data/cache/candidatures_profession.json"
with open(frontend_cache_path, "w", encoding="utf-8") as f:
    js.dump(json, f, ensure_ascii=False, indent=2)

## Vérification de l'exécution du fichier

In [ ]:
print("candidatures_profession exécuté")